# Synthetic Workplace Survey Analysis and Modelling

## Dataset Loading and Initial Inspection

This notebook presents a supplementary analysis using a synthetic workplace mental-health dataset. The synthetic dataset is not used as empirical evidence about real workers and does not replace the independently analysed real-world datasets.

Instead, it is used to examine the application of the common analytical framework to a larger, cleanly structured dataset containing workplace, wellbeing and burnout-related variables.

The dataset is analysed independently using the same general framework applied to the real-world datasets, including outcome assessment, predictor mapping, preprocessing, exploratory analysis, interpretable machine-learning modelling, performance evaluation and model interpretation.

In [2]:
import pandas as pd
import numpy as np

# Load the synthetic workplace survey dataset
synthetic = pd.read_csv('/Users/cristinaabrahamse/Documents/Eduvos University/Data Science Honours /Early Prediction of Burnout (Honours Proj.)/Burnout Datasets (Raw)/mental_health_workplace_survey.csv')

print("Dataset shape:", synthetic.shape)

print("\nColumns:")
print(synthetic.columns.tolist())

print("\nFirst five rows:")
display(synthetic.head())

Dataset shape: (3000, 25)

Columns:
['EmployeeID', 'Age', 'Gender', 'Country', 'JobRole', 'Department', 'YearsAtCompany', 'WorkHoursPerWeek', 'RemoteWork', 'BurnoutLevel', 'JobSatisfaction', 'StressLevel', 'ProductivityScore', 'SleepHours', 'PhysicalActivityHrs', 'CommuteTime', 'HasMentalHealthSupport', 'ManagerSupportScore', 'HasTherapyAccess', 'MentalHealthDaysOff', 'SalaryRange', 'WorkLifeBalanceScore', 'TeamSize', 'CareerGrowthScore', 'BurnoutRisk']

First five rows:


,EmployeeID,Age,Gender,Country,JobRole,Department,YearsAtCompany,WorkHoursPerWeek,RemoteWork,BurnoutLevel,...,CommuteTime,HasMentalHealthSupport,ManagerSupportScore,HasTherapyAccess,MentalHealthDaysOff,SalaryRange,WorkLifeBalanceScore,TeamSize,CareerGrowthScore,BurnoutRisk
0,1001,50,Male,UK,Sales Associate,HR,14,47,No,3.37,...,117,No,3.15,Yes,8,40K-60K,8.82,6,9.20,0
1,1002,36,Male,Germany,Software Engineer,IT,1,59,Hybrid,7.39,...,8,Yes,4.40,Yes,4,80K-100K,2.80,45,8.46,1
2,1003,29,Non-binary,India,IT Admin,IT,13,59,Hybrid,7.10,...,75,No,3.63,No,6,80K-100K,7.28,7,7.96,1
3,1004,42,Male,Australia,HR Specialist,IT,15,31,Yes,4.18,...,43,Yes,4.50,Yes,9,60K-80K,1.31,11,8.90,0
4,1005,40,Male,Brazil,Customer Support,Support,6,34,Yes,8.28,...,58,Yes,5.51,Yes,6,<40K,1.17,18,8.88,1


## Dataset Structure and Data Quality

The structure and quality of the synthetic dataset were assessed before variable selection and outcome construction. This included examination of data types, missing values, duplicate observations and unique values to identify any preprocessing requirements.

In [3]:
# Inspect dataset structure and data quality

print("Dataset dimensions:")
print(f"Rows: {synthetic.shape[0]}")
print(f"Columns: {synthetic.shape[1]}")

print("\nData types:")
print(synthetic.dtypes)

print("\nTotal missing values:")
print(synthetic.isnull().sum().sum())

print("\nMissing values by column:")
print(synthetic.isnull().sum()[synthetic.isnull().sum() > 0])

print("\nDuplicate rows:")
print(synthetic.duplicated().sum())

print("\nUnique values per column:")
print(synthetic.nunique())

Dataset dimensions:
Rows: 3000
Columns: 25

Data types:
EmployeeID                  int64
Age                         int64
Gender                     object
Country                    object
JobRole                    object
Department                 object
YearsAtCompany              int64
WorkHoursPerWeek            int64
RemoteWork                 object
BurnoutLevel              float64
JobSatisfaction           float64
StressLevel               float64
ProductivityScore         float64
SleepHours                float64
PhysicalActivityHrs       float64
CommuteTime                 int64
HasMentalHealthSupport     object
ManagerSupportScore       float64
HasTherapyAccess           object
MentalHealthDaysOff         int64
SalaryRange                object
WorkLifeBalanceScore      float64
TeamSize                    int64
CareerGrowthScore         float64
BurnoutRisk                 int64
dtype: object

Total missing values:
0

Missing values by column:
Series([], dtype: int64)

Du

## Variable and Outcome Assessment

Before selecting the modelling outcome and predictors, the variables were examined to understand their distributions, coding and conceptual roles.

Particular attention was given to `BurnoutLevel` and `BurnoutRisk` because the dataset contains both a continuous burnout measure and a supplied binary burnout-risk classification. Their relationship was assessed before determining the final outcome used for modelling and to prevent potential target leakage.

In [4]:
# Inspect categorical values and numerical ranges

categorical_columns = synthetic.select_dtypes(include="object").columns

print("CATEGORICAL VARIABLES")
print("=" * 50)

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(synthetic[column].value_counts())


numerical_columns = synthetic.select_dtypes(include=np.number).columns

print("\n\nNUMERICAL VARIABLE RANGES")
print("=" * 50)

numerical_summary = synthetic[numerical_columns].agg(
    ["min", "max", "mean", "std"]
).T

display(numerical_summary.round(2))

CATEGORICAL VARIABLES

--- Gender ---
Gender
Non-binary           757
Prefer not to say    757
Male                 752
Female               734
Name: count, dtype: int64

--- Country ---
Country
India        464
Brazil       443
Canada       434
UK           427
Australia    418
USA          408
Germany      406
Name: count, dtype: int64

--- JobRole ---
JobRole
Data Scientist       411
Software Engineer    401
Customer Support     380
Marketing Manager    372
Project Manager      362
HR Specialist        361
IT Admin             358
Sales Associate      355
Name: count, dtype: int64

--- Department ---
Department
HR             525
Sales          522
Marketing      521
Engineering    488
IT             482
Support        462
Name: count, dtype: int64

--- RemoteWork ---
RemoteWork
Hybrid    1022
No         990
Yes        988
Name: count, dtype: int64

--- HasMentalHealthSupport ---
HasMentalHealthSupport
No     1531
Yes    1469
Name: count, dtype: int64

--- HasTherapyAccess ---
HasT

,min,max,mean,std
EmployeeID,1001.0,4000.00,2500.50,866.17
Age,22.0,59.00,40.81,11.01
YearsAtCompany,0.0,20.00,10.10,6.04
WorkHoursPerWeek,30.0,59.00,44.50,8.49
BurnoutLevel,1.0,10.00,5.51,2.57
JobSatisfaction,1.0,10.00,5.44,2.59
StressLevel,1.0,10.00,5.52,2.60
ProductivityScore,1.0,10.00,5.52,2.61
SleepHours,4.0,9.00,6.54,1.44
PhysicalActivityHrs,0.0,10.00,5.03,2.86


### Burnout Outcome Verification

The dataset contains both a continuous `BurnoutLevel` score and a binary `BurnoutRisk` variable. Their relationship was examined to determine whether the supplied risk classification was derived from the continuous burnout score and to establish the most appropriate outcome for subsequent modelling.

This verification is important to ensure that variables directly used to define the modelling outcome are not subsequently included as predictors.

In [5]:
# Examine the relationship between BurnoutLevel and BurnoutRisk

print("Burnout-risk distribution:")
print(synthetic["BurnoutRisk"].value_counts().sort_index())

print("\nBurnout-risk percentages:")
print(
    synthetic["BurnoutRisk"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(1)
)

print("\nBurnoutLevel by BurnoutRisk:")
display(
    synthetic.groupby("BurnoutRisk")["BurnoutLevel"]
    .agg(["count", "min", "max", "mean", "median"])
    .round(2)
)

print("\nHighest BurnoutLevel classified as lower risk:")
print(
    synthetic.loc[
        synthetic["BurnoutRisk"] == 0,
        "BurnoutLevel"
    ].max()
)

print("\nLowest BurnoutLevel classified as elevated risk:")
print(
    synthetic.loc[
        synthetic["BurnoutRisk"] == 1,
        "BurnoutLevel"
    ].min()
)

Burnout-risk distribution:
BurnoutRisk
0    2020
1     980
Name: count, dtype: int64

Burnout-risk percentages:
BurnoutRisk
0    67.3
1    32.7
Name: proportion, dtype: float64

BurnoutLevel by BurnoutRisk:


,count,min,max,mean,median
BurnoutRisk,,,,,
0,2020,1.00,7.0,4.05,4.06
1,980,7.01,10.0,8.52,8.57



Highest BurnoutLevel classified as lower risk:
7.0

Lowest BurnoutLevel classified as elevated risk:
7.01


**Outcome verification:**  
The supplied `BurnoutRisk` variable contained 2,020 lower-risk observations (67.3%) and 980 elevated-risk observations (32.7%). Examination of the continuous `BurnoutLevel` variable showed a complete separation between the two classes: lower-risk observations had burnout scores ranging from 1.00 to 7.00, while elevated-risk observations ranged from 7.01 to 10.00.

This indicates that the supplied binary risk variable was directly derived from the continuous burnout score using a threshold above 7.00. The existing `BurnoutRisk` variable was therefore retained as the modelling outcome rather than constructing an additional percentile-based risk classification.

Because `BurnoutLevel` directly determines the binary outcome, it was excluded from the predictor set to prevent target leakage.

## LLM-Assisted Predictor Mapping and Researcher Validation

The dataset was already presented in English; therefore, language translation was not required. Variable names, data types, observed values and conceptual meaning were reviewed before modelling.

Consistent with the common analytical framework, LLM-assisted semantic mapping was used to organise variables into conceptual groups and propose their analytical roles. All proposed mappings were subsequently reviewed and validated by the researcher.

`EmployeeID` was treated as an identifier and excluded. `BurnoutRisk` was retained as the modelling outcome, while `BurnoutLevel` was excluded from the predictor set because it directly determines the binary outcome and would introduce target leakage.

In [6]:
# Create predictor mapping and audit table

predictor_audit = pd.DataFrame([
    ["EmployeeID", "Identifier", "Exclude",
     "Participant identifier"],

    ["Age", "Demographics", "Candidate predictor",
     "Participant age"],

    ["Gender", "Demographics", "Candidate predictor",
     "Gender category"],

    ["Country", "Demographics", "Candidate predictor",
     "Country of participant"],

    ["JobRole", "Employment characteristics", "Candidate predictor",
     "Occupational role"],

    ["Department", "Employment characteristics", "Candidate predictor",
     "Department of employment"],

    ["YearsAtCompany", "Employment characteristics", "Candidate predictor",
     "Length of employment at company"],

    ["WorkHoursPerWeek", "Work demands", "Candidate predictor",
     "Weekly working hours"],

    ["RemoteWork", "Work arrangement", "Candidate predictor",
     "Remote, hybrid or non-remote work arrangement"],

    ["BurnoutLevel", "Burnout outcome", "Exclude",
     "Continuous burnout score directly determining BurnoutRisk"],

    ["JobSatisfaction", "Workplace wellbeing", "Candidate predictor",
     "Job satisfaction score"],

    ["StressLevel", "Workplace wellbeing", "Candidate predictor",
     "Stress level score"],

    ["ProductivityScore", "Work performance", "Candidate predictor",
     "Productivity score"],

    ["SleepHours", "Wellbeing", "Candidate predictor",
     "Average sleep duration"],

    ["PhysicalActivityHrs", "Wellbeing", "Candidate predictor",
     "Physical activity duration"],

    ["CommuteTime", "Work characteristics", "Candidate predictor",
     "Commute duration"],

    ["HasMentalHealthSupport", "Workplace support", "Candidate predictor",
     "Availability of workplace mental-health support"],

    ["ManagerSupportScore", "Workplace support", "Candidate predictor",
     "Manager support score"],

    ["HasTherapyAccess", "Mental-health support", "Candidate predictor",
     "Access to therapy"],

    ["MentalHealthDaysOff", "Mental-health indicator", "Candidate predictor",
     "Mental-health-related days off"],

    ["SalaryRange", "Employment characteristics", "Candidate predictor",
     "Salary category"],

    ["WorkLifeBalanceScore", "Workplace wellbeing", "Candidate predictor",
     "Work-life balance score"],

    ["TeamSize", "Employment characteristics", "Candidate predictor",
     "Number of employees in team"],

    ["CareerGrowthScore", "Employment characteristics", "Candidate predictor",
     "Perceived career growth score"],

    ["BurnoutRisk", "Burnout outcome", "Outcome",
     "Supplied binary burnout-risk classification"]
], columns=[
    "Variable",
    "Conceptual_Group",
    "Proposed_Role",
    "Reason"
])

display(predictor_audit)

,Variable,Conceptual_Group,Proposed_Role,Reason
0,EmployeeID,Identifier,Exclude,Participant identifier
1,Age,Demographics,Candidate predictor,Participant age
2,Gender,Demographics,Candidate predictor,Gender category
3,Country,Demographics,Candidate predictor,Country of participant
4,JobRole,Employment characteristics,Candidate predictor,Occupational role
5,Department,Employment characteristics,Candidate predictor,Department of employment
6,YearsAtCompany,Employment characteristics,Candidate predictor,Length of employment at company
7,WorkHoursPerWeek,Work demands,Candidate predictor,Weekly working hours
8,RemoteWork,Work arrangement,Candidate predictor,"Remote, hybrid or non-remote work arrangement"
9,BurnoutLevel,Burnout outcome,Exclude,Continuous burnout score directly determining ...


In [7]:
# Record researcher validation of the proposed variable mappings

predictor_audit["Researcher_Validated"] = "Yes"
predictor_audit["Final_Role"] = predictor_audit["Proposed_Role"]

display(predictor_audit)

,Variable,Conceptual_Group,Proposed_Role,Reason,Researcher_Validated,Final_Role
0,EmployeeID,Identifier,Exclude,Participant identifier,Yes,Exclude
1,Age,Demographics,Candidate predictor,Participant age,Yes,Candidate predictor
2,Gender,Demographics,Candidate predictor,Gender category,Yes,Candidate predictor
3,Country,Demographics,Candidate predictor,Country of participant,Yes,Candidate predictor
4,JobRole,Employment characteristics,Candidate predictor,Occupational role,Yes,Candidate predictor
5,Department,Employment characteristics,Candidate predictor,Department of employment,Yes,Candidate predictor
6,YearsAtCompany,Employment characteristics,Candidate predictor,Length of employment at company,Yes,Candidate predictor
7,WorkHoursPerWeek,Work demands,Candidate predictor,Weekly working hours,Yes,Candidate predictor
8,RemoteWork,Work arrangement,Candidate predictor,"Remote, hybrid or non-remote work arrangement",Yes,Candidate predictor
9,BurnoutLevel,Burnout outcome,Exclude,Continuous burnout score directly determining ...,Yes,Exclude


## Data Preprocessing and Cleaning

Following variable assessment and researcher validation, a processed copy of the synthetic dataset was created for subsequent exploratory analysis and modelling.

The dataset contained no missing values or duplicate observations and did not require language translation or category correction. `EmployeeID` was excluded as an identifier. `BurnoutLevel` was retained for descriptive analysis but excluded from the modelling predictor set because it directly determines the supplied `BurnoutRisk` outcome.

In [8]:
# Create processed copy for subsequent analysis

workplace_synthetic = synthetic.copy()

# EmployeeID is not analytically meaningful
workplace_synthetic = workplace_synthetic.drop(
    columns=["EmployeeID"]
)

print("Processed dataset shape:", workplace_synthetic.shape)
print("Missing values:", workplace_synthetic.isnull().sum().sum())
print("Duplicate rows:", workplace_synthetic.duplicated().sum())

Processed dataset shape: (3000, 24)
Missing values: 0
Duplicate rows: 0


## Pre-Modelling Exploratory Data Analysis

Exploratory analysis was conducted before model development to examine the distribution of candidate predictors and their relationships with the supplied burnout-risk outcome.

Particular attention was given to workplace demands, wellbeing, support and performance-related variables to identify descriptive patterns associated with elevated burnout risk. These relationships are exploratory and do not imply causality.

In [9]:
# Compare key continuous predictors by burnout-risk group

eda_variables = [
    "WorkHoursPerWeek",
    "JobSatisfaction",
    "StressLevel",
    "ProductivityScore",
    "SleepHours",
    "PhysicalActivityHrs",
    "ManagerSupportScore",
    "MentalHealthDaysOff",
    "WorkLifeBalanceScore",
    "CareerGrowthScore"
]

risk_group_summary = (
    workplace_synthetic
    .groupby("BurnoutRisk")[eda_variables]
    .mean()
    .T
)

risk_group_summary.columns = [
    "Lower Risk Mean",
    "Elevated Risk Mean"
]

risk_group_summary["Difference"] = (
    risk_group_summary["Elevated Risk Mean"]
    - risk_group_summary["Lower Risk Mean"]
)

display(risk_group_summary.round(2))

,Lower Risk Mean,Elevated Risk Mean,Difference
WorkHoursPerWeek,44.61,44.29,-0.31
JobSatisfaction,5.45,5.42,-0.03
StressLevel,5.51,5.52,0.01
ProductivityScore,5.54,5.48,-0.07
SleepHours,6.55,6.51,-0.04
PhysicalActivityHrs,5.06,4.98,-0.08
ManagerSupportScore,5.48,5.37,-0.11
MentalHealthDaysOff,4.55,4.52,-0.03
WorkLifeBalanceScore,5.44,5.47,0.03
CareerGrowthScore,5.53,5.51,-0.02


### Numerical Predictor Relationships

Correlations among the numerical variables were examined to assess whether the synthetic workplace predictors showed meaningful relationships with burnout level and burnout risk before model development.

In [10]:
# Examine correlations with burnout outcomes

numeric_data = workplace_synthetic.select_dtypes(
    include=np.number
)

burnout_correlations = (
    numeric_data
    .corr()[["BurnoutLevel", "BurnoutRisk"]]
    .drop(index=["BurnoutLevel", "BurnoutRisk"])
    .sort_values(
        "BurnoutRisk",
        key=abs,
        ascending=False
    )
)

display(burnout_correlations.round(3))

,BurnoutLevel,BurnoutRisk
CommuteTime,0.005,0.025
ManagerSupportScore,-0.029,-0.020
WorkHoursPerWeek,-0.022,-0.017
PhysicalActivityHrs,-0.019,-0.013
SleepHours,-0.003,-0.013
ProductivityScore,-0.020,-0.012
YearsAtCompany,-0.013,-0.008
MentalHealthDaysOff,-0.004,-0.005
JobSatisfaction,-0.010,-0.005
WorkLifeBalanceScore,0.008,0.005


In [11]:
# Compare burnout-risk rates across categorical predictors

categorical_predictors = [
    "Gender",
    "Country",
    "JobRole",
    "Department",
    "RemoteWork",
    "HasMentalHealthSupport",
    "HasTherapyAccess",
    "SalaryRange"
]

for column in categorical_predictors:

    summary = (
        workplace_synthetic
        .groupby(column)["BurnoutRisk"]
        .agg(["count", "mean"])
        .rename(columns={
            "count": "Participants",
            "mean": "Elevated_Risk_Rate"
        })
    )

    summary["Elevated_Risk_Rate"] = (
        summary["Elevated_Risk_Rate"] * 100
    ).round(1)

    print(f"\n--- {column} ---")
    print(summary.sort_values(
        "Elevated_Risk_Rate",
        ascending=False
    ))


--- Gender ---
                   Participants  Elevated_Risk_Rate
Gender                                             
Prefer not to say           757                33.4
Non-binary                  757                32.9
Male                        752                32.8
Female                      734                31.5

--- Country ---
           Participants  Elevated_Risk_Rate
Country                                    
Germany             406                34.5
Canada              434                33.9
India               464                33.6
Brazil              443                32.3
UK                  427                32.1
Australia           418                31.1
USA                 408                31.1

--- JobRole ---
                   Participants  Elevated_Risk_Rate
JobRole                                            
HR Specialist               361                35.2
Software Engineer           401                35.2
Project Manager             362   